In [ ]:
# DATA TABLES
from module.core.ProjectDataset import Dataset

# RAW DATA 
data = Dataset(project="TCB2", filename="tissue_weight").select(experiment="default", region = 'cortex')
data.data

# QUANTATIVE STATS
data = (
    Dataset(project="TCB2", filename="tissue_weight")
    .select(experiment="default", region = 'cortex')
    .calculate_quantitative_statistics()
)
data.statistics_table 


# AGGREGATE STATS 
# data = (
#     Dataset(project="5HT4", filename="hplc") #saysfilename but hsould be dataset ie hplc / behavious / tissue_weight
#     .select(experiment="agonist_cortical5HT4")
#     .calculate_group_statistics()
# )
# df = data.group_statistics
# HACKY SAVE 
# df.to_excel('/Users/jasminebutler/Desktop/cloned_phd/PROJECTS/5HT4/group_stats.xlsx', index=False)

In [ ]:
# TABELS FOR EXPORT
from module.core.plotters import statistics_table

# QUANTITATIVE STATISTICS
statistics_table(
    project="5HT4",
    request={
        "datasets": {
            #  "tissue_weight":{ 
            #     "region":"TCB2_regions"
            # },
            "hplc": {
                "compound": ["DA", "NA", "5HT", "5HIAA", "5HTP", "DOPAC", "HVA"],  # dataset level selector
                "region": "5HT4_regions",
                "remove_outliers": {"grubbs": "calculated"},
            },
            # "behavior": {
            #     'measure':["white_floor","hessian_floor", "red_floor", "orange_floor","distance_traveled", "climbing", "sniffing","huddle"],
            #     "remove_outliers": {"grubbs": "calculated"}, 
            # },
        },
        "selector": {
            # "group": "vehicles"
            "experiment": "agonist_cortical5HT4",  # default  dose_response   agonist_antagonist agonist_cortical5HT4
        },
    },
)

In [ ]:
# HISTOGRAM
from module.core.plotters import histogram

data = histogram(
    project="5HT4",
    request={
        "datasets": {
            "hplc": {
                "compound": "DOPAC/DA",                 # dataset level selector
                # "compound": "neurotransmitters",      # dataset level selector
                "region": ["IL"],                       # dataset level selector
                # "region": "all_regions",
                "remove_outliers": {"grubbs":"calculated"}, 
                
            },
            # "behavior": {'measure':'lookaround', 
            #             #   "remove_outliers": "calculated",
            #              },  # dataset level selector
        },
        "selector": {                                   
            "experiment": "agonist_cortical5HT4",  #default takes all groups irrespective of experiment and runs stats (1 dependant vairable)
        },
    },
    # custom_params={"width": 15, "height": 120},)
)


data.statistics_table


In [ ]:
# SUMMARY HISTOGRAM
from module.core.plotters import summary_histogram

data = summary_histogram(
    project="5HT4", # TCB2
    request={
        "datasets": {
            "hplc": {
                "compound": ["5HIAA"],  # dataset level selector
                "region": "TCB2_regions",
                "remove_outliers": {"grubbs":"calculated"},
            },
            # "behavior": {
            #     'measure':'flooring_preference', #DeepOF_behavior distance_traveled flooring_preference
            #     "remove_outliers": {"grubbs":"calculated"},
            # },
        },
        "selector": {
            # "treatment": "vehicles"
            "experiment": "agonist_cortical5HT4", 
        },
    },
    # invert_hue=True,
    # custom_params={"fig_width": 20},
    # custom_params = {"palette": {'white_floor':'white', 'hessian_floor':'brown', 'red_floor':'red', 'orange_floor':'orange' }}

)


# data.statistics_table

In [ ]:
# CORRELOGRAM
from module.core.plotters import correlogram

data = correlogram(
    project="5HT4",
    request={
        "datasets": {
            "hplc": {
                "compound": ["DOPAC/DA"],              # dataset level selector: MUST SELECT FOR RATIOS
                # "compound": "neurotransmitters",      # dataset level selector
                # "region": ["OF"],                     # dataset level selector
                "region": "5HT4_regions",
                "remove_outliers": {"grubbs":"calculated"},
            },
        #     "behavior": {"flooring_preference"},      # dataset level selector: FOR CORR w BEHAVIOR and HPLC
        },
        "selector": {  
            "experiment": "agonist_cortical5HT4",  
        },
    },
between={"compound": [["DOPAC/DA", "DOPAC/DA"]]},       #  between={"dataset": [["hplc", "behavior"]]},

    # custom_params={
        # "p_value_threshold" : 0.2,
        # "fdr_correction": True,
        # # "width": 15, 
        # # "height": 120,
        # # "linewidths":0.8 # for edges on correlograms 
        # },
)


from module.core.plotters import correlogram




In [ ]:
# GENERATE COMPOUND PAIRS AND COLOR DICTS FOR NETWORK SUMMARY

def generate_combinations(compounds):
    # excluding same pairs and duplicates
    between_compounds = [[x, y] for i, x in enumerate(compounds) for y in compounds[i+1:]]
    # only same pairs
    within_compounds = [[x, x] for x in compounds]
    all_combinations = between_compounds + within_compounds
    return within_compounds, between_compounds, all_combinations

within_neurotransmitters, between_neurotransmitters, all__neurotransmitter_networks = generate_combinations(["GLU", "GABA", "ASP", "GLY", "TAU", "5HT", "DA", "NA"])

pal_between = {
    ('GLU', 'ASP'): 'lightgreen',
    ('GLU', 'GABA'): 'goldenrod',
    ('GLU', 'GLY'): 'olivedrab',
    ('GLU', 'TAU'): 'darkolivegreen',
    ('GLU', '5HT'): 'khaki',
    ('GLU', 'DA'): 'deepskyblue',
    ('GLU', 'NA'): 'cadetblue',
    ('GABA', 'ASP'): 'tomato',
    ('ASP', 'GLY'): 'rosybrown',
    ('ASP', 'TAU'): 'seagreen',
    ('ASP', '5HT'): 'wheat',
    ('ASP', 'DA'): 'slateblue',
    ('ASP', 'NA'): 'cornflowerblue',
    ('GABA', 'GLY'): 'darksalmon',
    ('GABA', 'TAU'): 'brown',
    ('GABA', '5HT'): 'peru',
    ('GABA', 'DA'): 'slategray',
    ('GABA', 'NA'): 'midnightblue',
    ('GLY', 'TAU'): 'darkkhaki',
    ('GLY', '5HT'): 'peachpuff',
    ('GLY', 'DA'): 'powderblue',
    ('GLY', 'NA'): 'dodgerblue',
    ('TAU', '5HT'): 'gold',
    ('TAU', 'DA'): 'goldenrod',  # Darkened for better visibility
    ('TAU', 'NA'): 'mediumslateblue',
    ('5HT', 'DA'): 'steelblue',  # Darkened for better contrast
    ('5HT', 'NA'): 'mediumaquamarine',
    ('DA', 'NA'): 'teal'
}
pal_within = {
        ('GLU', 'GLU'): 'limegreen',
        ('ASP', 'ASP'): 'mediumseagreen',
        ('GABA', 'GABA'): 'red',
        ('GLY', 'GLY'): 'indianred',
        ('TAU', 'TAU'): 'darkseagreen',
        ('5HT', '5HT'): 'orange',
        ('DA', 'DA'): 'steelblue',
        ('NA', 'NA'): 'purple'
    }

In [ ]:
# NETWORK SUMMARY
from module.core.plotters import network_summary

data = network_summary(
    project="TCB2",
    request={
        "datasets": {
            "hplc": {
                # "compound": "neurotransmitters",  # dataset level selector
                # "compound": "neurotransmitters",  # dataset level selector
                # "region": ["OF"],  # dataset level selector
                "region": "TCB2_regions",
                # "compound": "DA",
                "remove_outliers": {"grubbs":"calculated"},
            },
            # "behavior": {},  # dataset level selector
        },
        # "selector": {  # Apply to all datasets
            # "experiment": "agonist_antagonist",  # Stats will take this one into account
        # },
    },
    between={"compound": within_neurotransmitters},
    measurement="SD_node_degree",
    custom_params={"size": 10, "width": 10, "height": 10, "edgecolor": None, "palette": pal_within
                },  
                )


                #"density",
                # "total_edges",
                # "pos_edges",
                # "neg_edges",
                # "neg_pos_edge_ratio",
                # "max_degree",
                # "average_degree",
                # "min_degree",
                # "SD_node_degree",
                # "clust_coeff_unweighted",
                # "clust_coeff_weighted",
                # "global_efficiency",
                # "local_efficiency"

# name= between(network_classes.json) + measurement + region(region_classes.json)

In [ ]:
# NOT WORKING SUMMARY OF NETWOR SUMMARY
from module.core.plotters import summary_network_summary

data = summary_network_summary(
    project="TCB2",
    request={
        "datasets": {
            "hplc": {
                # "compound": "neurotransmitters",  # dataset level selector
                # "compound": "neurotransmitters",  # dataset level selector
                # "region": ["OF"],  # dataset level selector
                # "region": "all_regions",
                # "compound": "DA",
                "remove_outliers": {"grubbs":"calculated"},
            },
            # "behavior": {},  # dataset level selector
        },
        # "selector": {  # Apply to all datasets
        #     "experiment": "agonist_antagonist",  # Stats will take this one into account
        # },
    },
    between={"compound": [["DA", "5HT"], ["NA", "5HT"], ["GLU", "5HT"]]},
    # custom_params={"width": 15, "height": 120},)
)

In [ ]:
# NETWORK
from module.core.plotters import network

data = network(
    project="TCB2",
    request={
        "datasets": {
            "hplc": {
                "compound": ["GABA", "5HT"],        
                # "compound": "neurotransmitters",      
                # "region": ["OF"],                    
                "region": "TCB2_regions",
                "remove_outliers": {"grubbs":"calculated"},
            },
            # "behavior": {},                          
        },
        "selector": {  
            "experiment": "agonist_antagonist", 
        },
    },
    between={"compound": [["GABA", "5HT"]]},
    # custom_params={"node_position": "circle"}
)

In [ ]:
# NETWORK DEGREE DISTRIBUTION
from module.core.plotters import network_degrees

data = network_degrees(
    project="TCB2",
    request={
        "datasets": {
            "hplc": {
                "compound": "NA",  
                # "compound": "neurotransmitters",  
                # "region": ["OF"],  
                "region": "TCB2_regions",
                "remove_outliers": {"grubbs":"calculated"},
            },
            # "behavior": {},  
        },
        "selector": {  
            "experiment": "agonist_antagonist",  
        },
    },
    between={"compound": [["NA", "NA"]]
             },
    # custom_params={"width": 15, "height": 120},)
)